# Description

This code generate a co-citation graph of the corpus. The output are in PDF and PGF formats.

# Prerequisites

- matplotlib
- pandas
- openpyxl

- [grobid](https://grobid.readthedocs.io/en/latest/Grobid-docker/) to generate the tei.xml files of each doc

```
docker run --rm --gpus all --init --ulimit core=0 -p 8070:8070 grobid/grobid:0.9.0-full
```

In [41]:
import pandas as pd

import reference_man
import random
import numpy as np



In [42]:
filename = '../data/refs.xlsx'
sheet_name = 'RAW'
output_file = "co_citation_network"
seed = 42
random.seed(seed)
np.random.seed(seed)

In [43]:
corpus = pd.read_excel(filename, sheet_name=sheet_name)

In [44]:
refs = {}

In [45]:
corpus_titles = list(corpus['Title'])

In [46]:
refs_dict = {}

In [47]:
for id,item in corpus.iterrows():
    pub = reference_man.Publication.from_xlsx(item)
    pub_title = pub.get_title()
    refs[pub_title] = set()
    for reference in pub.get_references():
        ref_title = reference.get_title()
        if ref_title is not None:
            if ref_title not in refs:
                refs[ref_title] = set()

In [48]:
import Levenshtein


In [49]:
for id,item in corpus.iterrows():
    pub = reference_man.Publication.from_xlsx(item)
    pub_title = pub.get_title()
    for reference in pub.get_references():
        ref_title = reference.get_title()

In [50]:
for id,item in corpus.iterrows():
    pub = reference_man.Publication.from_xlsx(item)
    pub_title = pub.get_title()

    print("-----",pub_title)
    print("nb references=",len(pub.get_references()))
    refs_dict[pub_title] = pub.get_references()
    for reference in pub.get_references():
        ref_title = reference.get_title()
        if ref_title is not None:
            match = None
            for t in refs.keys():
                if Levenshtein.ratio(t,ref_title) > 0.75: 
                    match = t
                    break

            if match:
                refs[pub_title].add(match)

----- Climate change adaptation in Australian mining communities: comparing mining company and local government views and activities
nb references= 27
----- In Situ Adaptation to Climatic Change: Mineral Industry Responses to Extreme Flooding Events in Queensland, Australia
nb references= 66
----- Mining Communities from a Resilience Perspective: Managing Disturbance and Vulnerability in Itabira, Brazil
nb references= 40
----- Mining amid typhoons: Large-scale mining and typhoon vulnerability in the Philippines
nb references= 73
----- BUILDING COMMUNITY RESILIENCE IN MINE IMPACTED COMMUNITIES: A STUDY ON DELIVERY OF HEALTH SERVICES IN PAPUA NEW GUINEA
nb references= 467
----- Emerging grassroots resilience and flood responses in informal settlements in Accra, Ghana
nb references= 45
----- Mining community resilience explored through sustainable community development and perceptions of community wellbeing. 
nb references= 130
----- DISASTER GOVERNANCE AND COMMUNITY RESILIENCE:  THE LAW 

In [51]:
#use pfg

use_pfg = True

if use_pfg:
    import matplotlib as mpl

    mpl.use("pgf")
    mpl.rcParams.update({
        "pgf.texsystem": "pdflatex",   
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,    
    })


In [52]:
from matplotlib import pyplot as plt
import networkx as nx

from itertools import combinations

G = nx.Graph()

corpus_titles = set(refs.keys())

for id,item in corpus.iterrows():
    doc = item['Title']
    references = refs[doc]
    label_name = item["BiblCitation"]
    G.add_node(doc, type="corpus",label=label_name)

    for ref in references:
        if ref != doc:
            G.add_edge(doc, ref, weight=1)

labels = nx.get_node_attributes(G, "label")


print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())



Nodes: 1721
Edges: 1747


In [53]:
for corpus_node in [n for n, d in G.nodes(data=True) if d.get("type") == "corpus"]:
    neighbors = list(G.neighbors(corpus_node))

    leaf_neighbors = [n for n in neighbors if G.degree[n] == 1 and G.nodes[n].get("type") != "corpus"]

    keep = set(leaf_neighbors[:20])

    remove = [n for n in leaf_neighbors if n not in keep]
    G.remove_nodes_from(remove)

print("Graph final :", G.number_of_nodes(), "noeuds et", G.number_of_edges(), "arêtes")

Graph final : 542 noeuds et 568 arêtes


In [54]:
low_degree_nodes = [n for n, d in G.degree() if d <= 1]
G.remove_nodes_from(low_degree_nodes)


In [55]:
from adjustText import adjust_text

fig, ax = plt.subplots(figsize=(14, 10))

pos = nx.spring_layout(G, seed=10, k=0.55, iterations=100)

red_edges = [
    (u, v) for u, v in G.edges()
    if G.nodes[u].get("type") == "corpus" and G.nodes[v].get("type") == "corpus"
]
gray_edges = [
    (u, v) for u, v in G.edges()
    if (u, v) not in red_edges
]

nx.draw_networkx_edges(G, pos, edgelist=gray_edges, edge_color="gray",
                       alpha=0.4, width=0.6, arrowstyle='-', arrows=True,
                       connectionstyle="arc3,rad=0.1", ax=ax)

nx.draw_networkx_edges(G, pos, edgelist=red_edges, edge_color="red",
                       width=2, alpha=0.8, arrowstyle='-', arrows=True,
                       connectionstyle="arc3,rad=0.1", ax=ax)

corpus_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "corpus"]
ref_nodes    = [n for n, d in G.nodes(data=True) if d.get("type") != "corpus"]

nx.draw_networkx_nodes(G, pos, nodelist=ref_nodes, node_size=1, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=corpus_nodes, node_color="red", node_size=10, ax=ax)

texts = []
for node, label in labels.items():
    x, y = pos[node]
    t = ax.text(x, y, label, fontsize=12, va="bottom", ha="left")
    texts.append(t)

adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.4)
)

([Text(-0.4058230032871337, 0.14073980339766834, '\\cite{loechel_climate_2013}'),
  Text(-0.3413303470743414, 0.012462006914113255, '\\cite{sharma_situ_2013}'),
  Text(0.5092724848001973, 0.5071636580951302, '\\cite{wasylycia-leis_mining_2014} '),
  Text(0.8643675294566804, -0.05158426914011982, '\\cite{holden_mining_2015} '),
  Text(0.4202102414992255, 0.36626349826956983, '\\cite{kuir-ayius_building_2016}'),
  Text(0.387921540256013, -0.10661508246371643, '\\cite{amoako_emerging_2018} '),
  Text(0.1730530396900305, 0.5291914777354405, '\\cite{kanakis_mining_2018}'),
  Text(-0.8391414454357041, -0.6753637651849617, '\\cite{goyal_disaster_2019} '),
  Text(-0.5150686304776141, 0.06605745160333232, '\\cite{mavrommatis_impacts_2020}'),
  Text(0.7013891642041323, 0.06353004346791269, '\\cite{odell_desalination_2021} '),
  Text(0.7939756757877148, 0.3972841681049566, '\\cite{liu_dynamics_2021}'),
  Text(0.5695614439404646, 0.35343243135625424, '\\cite{irshad_landslide_2021} '),
  Text(-0.06

In [56]:

plt.savefig(output_file+".pgf")  
plt.savefig(output_file+".pdf")  


In [57]:
#pip install pyvis